In [3]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine, URL, text
import getpass

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

In [5]:
password = getpass.getpass("Enter PostgreSQL password: ")

connection_url = URL.create(
    drivername="postgresql+psycopg",
    username="postgres",
    password=password,
    host="localhost",
    port=5432,
    database="olist_analysis"
)

engine = create_engine(connection_url)

with engine.connect() as connection:
    print("PostgreSQL connection successful!")

Enter PostgreSQL password:  ········


PostgreSQL connection successful!


In [31]:
tables = [
    "orders",
    "customers",
    "order_items",
    "products",
    "sellers",
    "payments",
    "reviews"
]

data = {}

for table in tables:
    data[table] = pd.read_sql_table(
        table,
        con=engine,
        schema="public"
    )

    print(
        f"{table}: "
        f"{data[table].shape[0]:,} rows × "
        f"{data[table].shape[1]} columns"
    )

orders: 99,441 rows × 8 columns
customers: 99,441 rows × 5 columns
order_items: 112,650 rows × 7 columns
products: 32,951 rows × 9 columns
sellers: 3,095 rows × 4 columns
payments: 103,886 rows × 5 columns
reviews: 99,224 rows × 7 columns


In [21]:
initial_report = []

for name, df in data.items():

    initial_report.append({
        "table": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_values": df.isna().sum().sum(),
        "duplicate_rows": df.duplicated().sum()
    })

initial_report = pd.DataFrame(initial_report)

initial_report

,table,rows,columns,missing_values,duplicate_rows
0,orders,99441,8,4908,0
1,customers,99441,5,0,0
2,order_items,112650,7,0,0
3,products,32951,9,1838,0
4,sellers,3095,4,0,0
5,payments,103886,5,0,0
6,reviews,99224,7,145903,0


In [22]:
duplicate_report = []

for name, df in data.items():

    duplicate_count = df.duplicated().sum()

    duplicate_report.append({
        "table": name,
        "duplicate_rows": duplicate_count
    })

    data[name] = df.drop_duplicates().copy()

duplicate_report = pd.DataFrame(duplicate_report)

duplicate_report

,table,duplicate_rows
0,orders,0
1,customers,0
2,order_items,0
3,products,0
4,sellers,0
5,payments,0
6,reviews,0


In [23]:
reviews = data["reviews"]

duplicate_review_ids = reviews[
    reviews.duplicated(subset=["review_id"], keep=False)
]

print(
    "Rows belonging to duplicated review IDs:",
    len(duplicate_review_ids)
)

print(
    "Number of duplicated review IDs:",
    duplicate_review_ids["review_id"].nunique()
)

Rows belonging to duplicated review IDs: 1603
Number of duplicated review IDs: 789


In [24]:
review_conflicts = (
    duplicate_review_ids
    .groupby("review_id")
    .nunique(dropna=False)
)

conflicting_review_ids = review_conflicts[
    review_conflicts.gt(1).any(axis=1)
]

print(
    "Review IDs with conflicting information:",
    len(conflicting_review_ids)
)

Review IDs with conflicting information: 789


In [12]:
if len(conflicting_review_ids) == 0:

    reviews = reviews.drop_duplicates(
        subset=["review_id"],
        keep="first"
    ).copy()

    data["reviews"] = reviews

    print("Duplicate reviews successfully removed.")

else:
    raise ValueError(
        "Some duplicated review IDs contain conflicting information. "
        "Investigate before deleting."
    )

Duplicate reviews successfully removed.


In [28]:
duplicated_reviews = reviews[
    reviews["review_id"].duplicated(keep=False)
].sort_values("review_id")

print(f"Rows involved: {len(duplicated_reviews):,}")
print(f"Distinct duplicated IDs: {duplicated_reviews['review_id'].nunique():,}")

display(duplicated_reviews.tail(20))

Rows involved: 1,603
Distinct duplicated IDs: 789


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
89366,fb96ea2ef8cce1c888f4d45c8e22b793,c88b1d1b157a9999ce368f218a407141,5,None,None,2017-07-21,2017-07-26 13:45:15
9377,fb96ea2ef8cce1c888f4d45c8e22b793,38beac7cca7c937da89f8aedee76f944,5,None,None,2017-07-21,2017-07-26 13:45:15
67653,fbb7c6f69326ad5bf986c099bedefdb4,d351ad232c9e018edafa9f34f5d03e63,1,não recomendo,produto não veio conforme o comprado.,2018-08-08,2018-08-10 13:14:46
29248,fbb7c6f69326ad5bf986c099bedefdb4,4608a27e9e60c05b054631def491e7af,1,não recomendo,produto não veio conforme o comprado.,2018-08-08,2018-08-10 13:14:46
39030,fc906850133b4a339e3aad7b79dbfa7a,f811e0db5991b2b39a1fe8a057fc6f96,1,Não recebi até 28/04/20,None,2018-04-20,2018-04-28 12:44:12
11912,fc906850133b4a339e3aad7b79dbfa7a,c711d7f26c4563f427379e7ed3bfdc51,1,Não recebi até 28/04/20,None,2018-04-20,2018-04-28 12:44:12
79408,fd53ace363f9af2bc44ae028b2e926d6,c66985d27ca5899e6058749d3432daf4,5,None,None,2017-08-16,2017-08-19 10:34:40
35076,fd53ace363f9af2bc44ae028b2e926d6,b6f999c6eccbf4df9af77d67c77a8e52,5,None,None,2017-08-16,2017-08-19 10:34:40
7055,fd582f520c76d0b29106fcef19d868fc,46ad8f532d3f76a1747288a6a5a88a18,4,None,None,2017-10-26,2017-10-30 14:55:42
92071,fd582f520c76d0b29106fcef19d868fc,78668b283d34d8c72670a44109fbbe09,4,None,None,2017-10-26,2017-10-30 14:55:42


In [29]:
# Get all duplicated review IDs
duplicated_reviews = reviews[
    reviews.duplicated(
        subset="review_id",
        keep=False
    )
].copy()

# Count unique orders per duplicated review ID
review_id_analysis = (
    duplicated_reviews
    .groupby("review_id")
    .agg(
        number_of_records=("order_id", "size"),
        unique_orders=("order_id", "nunique"),
        unique_scores=("review_score", "nunique"),
        unique_creation_dates=("review_creation_date", "nunique"),
        unique_answer_timestamps=("review_answer_timestamp", "nunique")
    )
    .reset_index()
)

review_id_analysis.head(10)

,review_id,number_of_records,unique_orders,unique_scores,unique_creation_dates,unique_answer_timestamps
0,00130cbe1f9d422698c812ed8ded1919,2,2,1,1,1
1,0115633a9c298b6a98bcbe4eee75345f,2,2,1,1,1
2,0174caf0ee5964646040cd94e15ac95e,2,2,1,1,1
3,017808d29fd1f942d97e50184dfb4c13,2,2,1,1,1
4,0254bd905dc677a6078990aad3331a36,2,2,1,1,1
5,0288d42bef3dfe36930740c9588a570f,2,2,1,1,1
6,02aa7f5f75e964e3c7efa59a1f515281,2,2,1,1,1
7,034528cb00b6bc981847acafbdf2ae0b,2,2,1,1,1
8,03a6a25db577d0689440933055111897,2,2,1,1,1
9,0467560f511c516ddaa54a60edb0c291,2,2,1,1,1


In [36]:
orders = data["orders"]
customers = data["customers"]
review_order_check = duplicated_reviews.merge(
    orders[["order_id", "order_status", "customer_id"]],
    on="order_id",
    how="left"
)

review_order_check = review_order_check.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
).sort_values("review_id")

review_order_check.head(20)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,order_status,customer_id,customer_unique_id
754,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,None,O cartucho original HP 60XL não é reconhecido ...,2018-03-07,2018-03-20 18:08:23,delivered,a7026133ddbd2e86c83ecd4dfa4dbe01,f30856ad31d3e74253a3f4ccef670648
464,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,None,O cartucho original HP 60XL não é reconhecido ...,2018-03-07,2018-03-20 18:08:23,delivered,fef2e5e63da9f3e1dd89e8e319468657,f30856ad31d3e74253a3f4ccef670648
1460,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,None,None,2017-09-21,2017-09-26 03:27:47,delivered,d39c97549a412a82dacf30fe5f144cb0,91ffaf5eb7f4bd6a48b07f3546fb7a99
1017,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,None,None,2017-09-21,2017-09-26 03:27:47,delivered,3bb15be93abf3c2a684d163ba6faaa5b,91ffaf5eb7f4bd6a48b07f3546fb7a99
1498,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,None,Produto entregue dentro de embalagem do fornec...,2018-03-07,2018-03-08 03:00:53,delivered,ef9e24d18ac44872aa39e9bb721f9880,ba16ab3721ffd6b57fc87a7a4f6a4fcf
931,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,None,Produto entregue dentro de embalagem do fornec...,2018-03-07,2018-03-08 03:00:53,delivered,5043e611d34f6ad724ca658aa22cc43c,ba16ab3721ffd6b57fc87a7a4f6a4fcf
897,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,None,None,2018-03-02,2018-03-05 01:43:30,delivered,ad3027360ab7ff592efb16fb347d1e1f,d5ca668597a3b40c1e527db8c761f03c
1601,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,None,None,2018-03-02,2018-03-05 01:43:30,delivered,46ce474c7e3c57c93e6bdf2d6e6162f2,d5ca668597a3b40c1e527db8c761f03c
315,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,None,O pedido consta de 2 produtos e até agora rece...,2017-09-09,2017-09-13 09:52:44,delivered,573f39624e482d14affea43eb0588fcb,4c45ebf94ecf4107850d91bc326851c3
1553,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,None,O pedido consta de 2 produtos e até agora rece...,2017-09-09,2017-09-13 09:52:44,delivered,576230a5830c1bebb06d2ee474e79555,4c45ebf94ecf4107850d91bc326851c3


In [37]:
reviews = data["reviews"].copy()

print("Rows before cleaning:", len(reviews))

reviews = reviews.drop_duplicates(
    subset=["review_id"],
    keep="first"
).copy()

print("Rows after cleaning:", len(reviews))

print(
    "Removed rows:",
    len(data["reviews"]) - len(reviews)
)

data["reviews"] = reviews

Rows before cleaning: 99224
Rows after cleaning: 98410
Removed rows: 814


In [38]:
print(
    "Remaining duplicated review IDs:",
    reviews["review_id"].duplicated().sum()
)

print(
    "Remaining duplicated rows:",
    reviews.duplicated().sum()
)

Remaining duplicated review IDs: 0
Remaining duplicated rows: 0


In [40]:
with engine.begin() as connection:
    connection.execute(
        text("CREATE SCHEMA IF NOT EXISTS clean")
    )

print("Clean schema created successfully!")

Clean schema created successfully!


In [42]:
reviews.to_sql(
    name="reviews",
    con=engine,
    schema="clean",
    if_exists="replace",
    index=False,
    chunksize=5000
)

-20